# Tutorial #1 - Manual MLOps Workflow

This notebook demonstrates the manual DVC + MLflow integration pattern:

1. Load the Jena Climate dataset (already tracked by DVC)
2. Read the DVC content hash from the `.dvc` pointer file
3. Log the hash as an MLflow parameter and tag for strict data lineage
4. Train a simple baseline model and log metrics

All steps use standard Python code and CLI tools - no platform-specific features.

## 1. Data Ingestion

The Jena Climate dataset is hosted in the TensorFlow/Keras dataset repository. The ingestion script downloads the compressed file, extracts it, and places the CSV in the `data/` folder. Once ingested, we track it with DVC to enable reproducible versioning.

In [2]:
import os
import keras
from zipfile import ZipFile

data_dir = os.path.join(os.getcwd(), "../data")
csv_path = os.path.join(data_dir, "jena_climate_2009_2016.csv")

if not os.path.exists(csv_path):
    uri = "https://storage.googleapis.com/tensorflow/tf-keras-datasets/jena_climate_2009_2016.csv.zip"
    zip_path = keras.utils.get_file(origin=uri, fname="jena_climate_2009_2016.csv.zip")
    ZipFile(zip_path).extractall(data_dir)
    print(f"Downloaded and extracted to {csv_path}")
else:
    print(f"Dataset already exists at {csv_path}")

print(f"File size: {os.path.getsize(csv_path) / 1024 / 1024:.1f} MB")

<>:5: SyntaxWarning: invalid escape sequence '\d'
<>:5: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_493/1205567653.py:5: SyntaxWarning: invalid escape sequence '\d'
  data_dir = os.path.join(os.getcwd(), "..\data")


/tmp/ipykernel_493/1205567653.py:5: SyntaxWarning: invalid escape sequence '\d'
  data_dir = os.path.join(os.getcwd(), "..\data")


AttributeError: partially initialized module 'pandas' has no attribute 'core' (most likely due to a circular import)

### DVC Tracking

After ingestion, we track the dataset with DVC. This creates a `.dvc` pointer file containing the MD5 hash and adds the CSV to `.gitignore` (since DVC manages it via the remote storage, not Git).

In [2]:
!dvc add data/jena_climate_2009_2016.csv
!git add data/jena_climate_2009_2016.csv.dvc data/.gitignore
!dvc push

## 2. Dataset Loading

The Jena Climate dataset contains 420,551 observations across 15 columns (14 meteorological variables plus a timestamp), recorded every 10 minutes from January 2009 to December 2016.

In [3]:
import pandas as pd

df = pd.read_csv(csv_path)

print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"Date range: {df['Date Time'].iloc[0]} to {df['Date Time'].iloc[-1]}")
df.head()

## 3. DVC Data Lineage

The `.dvc` pointer file contains the MD5 hash that uniquely identifies this exact version of the data. We read this hash and will log it into MLflow to establish strict data lineage - every experiment run is linked to the precise dataset version used.

In [4]:
import yaml

dvc_file = csv_path + ".dvc"

with open(dvc_file) as f:
    dvc_meta = yaml.safe_load(f)

dvc_hash = dvc_meta["outs"][0]["md5"]
dvc_size = dvc_meta["outs"][0]["size"]

print(f"DVC MD5 hash: {dvc_hash}")
print(f"File size:    {dvc_size:,} bytes ({dvc_size / 1024 / 1024:.1f} MB)")

## 4. Preprocessing

Following the project guidelines, we select 6 input features (avoiding near-deterministic relationships with the target), resample to hourly frequency, and split temporally.

In [5]:
import numpy as np

# Parse dates and set as index
df["Date Time"] = pd.to_datetime(df["Date Time"], format="%d.%m.%Y %H:%M:%S")
df.set_index("Date Time", inplace=True)

# Replace erroneous wind speed values
df[["wv (m/s)", "max. wv (m/s)"]] = df[["wv (m/s)", "max. wv (m/s)"]].replace(-9999.0, np.nan)

# Remove duplicates
n_before = len(df)
df = df[~df.index.duplicated(keep="first")]
print(f"Removed {n_before - len(df)} duplicate rows")

# Resample to hourly (mean aggregation)
df = df.resample("1h").mean().dropna()
print(f"Hourly shape: {df.shape}")

# Select features
feature_cols = ["T (degC)", "p (mbar)", "rh (%)", "wv (m/s)", "max. wv (m/s)", "wd (deg)"]
target_col = "T (degC)"
df_model = df[feature_cols].copy()

# Temporal split (70% train, 15% val, 15% test)
n = len(df_model)
train_end = int(n * 0.7)
val_end = int(n * 0.85)

df_train = df_model.iloc[:train_end]
df_val = df_model.iloc[train_end:val_end]
df_test = df_model.iloc[val_end:]

print(f"Train: {len(df_train)}, Val: {len(df_val)}, Test: {len(df_test)}")

## 5. Baseline Model with MLflow Tracking

We train a simple linear regression as a baseline and log everything to MLflow: the DVC data hash (as both parameter and tag), model parameters, and evaluation metrics. This establishes the data lineage pattern that will be used with more complex models (GRU, PatchTST).

In [6]:
import mlflow
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Prepare features and target (1-step ahead: predict next hour's temperature)
X_train = df_train[feature_cols].iloc[:-1].values
y_train = df_train[target_col].iloc[1:].values

X_test = df_test[feature_cols].iloc[:-1].values
y_test = df_test[target_col].iloc[1:].values

with mlflow.start_run(run_name="baseline_linear_regression"):
    # Log DVC data hash - strict data lineage
    mlflow.log_param("dvc_data_hash", dvc_hash)
    mlflow.set_tag("dvc.data_hash", dvc_hash)
    mlflow.set_tag("dvc.data_file", "data/jena_climate_2009_2016.csv")

    # Log experiment parameters
    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_param("features", feature_cols)
    mlflow.log_param("target", target_col)
    mlflow.log_param("train_size", len(X_train))
    mlflow.log_param("test_size", len(X_test))
    mlflow.log_param("forecast_strategy", "1-step ahead")

    # Train
    model = LinearRegression()
    model.fit(X_train, y_train)

    # Evaluate
    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    mlflow.log_metric("mae", mae)
    mlflow.log_metric("rmse", rmse)

    print(f"MAE:  {mae:.4f} C")
    print(f"RMSE: {rmse:.4f} C")
    print(f"DVC hash logged: {dvc_hash}")

## 6. Verify Data Lineage in MLflow

We confirm the DVC hash was correctly logged by querying the last run.

In [7]:
runs = mlflow.search_runs(order_by=["start_time DESC"], max_results=1)
run = runs.iloc[0]

print(f"Run:           {run['tags.mlflow.runName']}")
print(f"DVC hash (param): {run['params.dvc_data_hash']}")
print(f"DVC hash (tag):   {run['tags.dvc.data_hash']}")
print(f"Data file:        {run['tags.dvc.data_file']}")
print(f"MAE:              {run['metrics.mae']:.4f}")
print(f"RMSE:             {run['metrics.rmse']:.4f}")